In [1]:
import ConnectionConfig as cc
from delta import DeltaTable
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("factRit")
spark.getActiveSession()

In [3]:
cc.set_connectionProfile("VeloDB")

df_operational_rides = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

#### 2 MAKE DIMENSION TABLES AVAILABLE AS VIEWS


In [4]:
dimDate = spark.read.format("delta").load("spark-warehouse/dimDate")
dimWeather = spark.read.format("delta").load("spark-warehouse/dimWeather")
dim_Customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLock")

dim_Customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
dimDate.createOrReplaceTempView("dimDate")
dimWeather.createOrReplaceTempView("dimWeather")

In [5]:
df_operational_rides.createOrReplaceTempView("rides_source")

In [6]:
# todo scd doen ## uuid neemt veel ruimte 
fact_rides = spark.sql("""
    SELECT 
        r.rideId AS rideId,
        s1.lockSK AS startLockSK,
        s2.lockSK AS endLockSK,
        r.starttime AS starttime,
        r.endtime AS endtime,
        c.userId AS userId,
        d.dateSK AS dateSK  -- Adding date dimension key
    FROM 
        rides_source AS r
    LEFT OUTER JOIN 
        dimLocks AS s1 ON r.startlockId = s1.lockId
    LEFT OUTER JOIN 
        dimLocks AS s2 ON r.endlockId = s2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON r.subscriptionid = c.userId
    LEFT OUTER JOIN 
        dimDate AS d ON date_format(r.starttime, 'yyyy-MM-dd') = d.calendarDate
    WHERE 
        s1.lockId IS NOT NULL AND 
        s2.lockId IS NOT NULL AND 
        c.userId IS NOT NULL
""")

fact_rides.show(20)
fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRides")




+------+--------------------+--------------------+-------------------+-------------------+------+------+
|rideId|         startLockSK|           endLockSK|          starttime|            endtime|userId|dateSK|
+------+--------------------+--------------------+-------------------+-------------------+------+------+
|     1|fd7ed20e-c9f3-4ac...|ebe161db-4ef9-407...|2015-09-22 00:00:00|2012-09-22 00:00:00| 13296|  2455|
|     3|651d3f9a-59bd-405...|aa987476-a4dd-4d6...|2015-09-22 00:00:00|2012-09-22 00:00:00| 25722|  2455|
|     4|5eb15062-efdd-428...|e481418c-2466-4a8...|2015-09-22 00:00:00|2012-09-22 00:00:00| 31000|  2455|
|     5|cc6a2651-a05e-40a...|ac023f7f-8563-4fc...|2015-09-22 00:00:00|2012-09-22 00:00:00| 59732|  2455|
|     7|5ce09ba2-447a-4c3...|dfc5bb4b-3cf0-4cc...|2015-09-22 00:00:00|2012-09-22 00:00:00| 31055|  2455|
|    11|eb5d77b6-e63e-419...|244e57c1-ce0c-447...|2015-09-22 00:00:00|2012-09-22 00:00:00|   999|  2455|
|    12|1faa90ca-a3a5-405...|172b8eb3-5a65-42a...|2015-

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [29]:
# Create the fact table `factRides`
#############Helemaal correct 
# fact_rides = spark.sql("""
#     SELECT 
#         r.rideId AS rideId,
#         s1.lockSK AS startLockSK,
#         s2.lockSK AS endLockSK,
#         r.starttime AS starttime,
#         r.endtime AS endtime,
#         c.userId AS userId
#     FROM 
#         rides_source AS r
#     LEFT OUTER JOIN 
#         dimLocks AS s1 ON r.startlockId = s1.lockId
#     LEFT OUTER JOIN 
#         dimLocks AS s2 ON r.endlockId = s2.lockId
#     LEFT OUTER JOIN 
#         dimCustomer AS c ON r.subscriptionid = c.userId
#     WHERE 
#         s1.lockId IS NOT NULL AND s2.lockId IS NOT NULL AND c.userId IS NOT NULL
# """)
# fact_rides.show(20)
# fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRides")

+------+--------------------+--------------------+-------------------+-------------------+------+
|rideId|         startLockSK|           endLockSK|          starttime|            endtime|userId|
+------+--------------------+--------------------+-------------------+-------------------+------+
|     1|dcf9b1dd-36ef-475...|b3cdea99-59db-40e...|2019-09-22 08:46:43|2019-09-22 09:01:36| 13296|
|     3|d27d09d3-a7ee-457...|3e92973c-b615-464...|2019-09-22 08:27:38|2019-09-22 08:30:25| 25722|
|     4|91d2992b-10b5-488...|e20596cb-86f6-41b...|2019-09-22 08:41:48|2019-09-22 08:46:52| 31000|
|     5|a12d40fc-16ed-439...|13e102b4-3201-45c...|2019-09-22 08:50:08|2019-09-22 09:09:02| 59732|
|     7|61e8340b-cce6-470...|85a891ad-3149-437...|2019-09-22 08:05:17|2019-09-22 08:14:44| 31055|
|    11|ba4019e9-294f-4fc...|12466ef5-fa87-40f...|2019-09-22 08:48:14|2019-09-22 08:52:45|   999|
|    12|6f316db6-d718-444...|856ad9e1-ef1e-45a...|2019-09-22 08:14:52|2019-09-22 08:25:33| 59847|
|    13|d00e2582-d53

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\KDG 2024-25\Data Analytics\sparkdelta\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ssaai\AppData\Local\Temp\ipykernel_6408\3535403226.py", line 22, in <module>
    fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRides")
  File "C:\KDG 2024-25\Data Analytics\sparkdelta\venv\Lib\site-packages\pyspark\sql\readwriter.py", line 1586, in saveAsTable
    self._jwrite.saveAsTable(name)
  File "C:\KDG 2024-25\Data Analytics\sparkdelta\venv\Lib\site-packages\py4j\java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "C:\KDG 2024-25\Data Analytics\sparkdelta\venv\Lib\site-packages\pyspark\errors\exceptions\captured.py", line 179, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "C:\KDG 2024-25\Data Analyt

ConnectionRefusedError: [WinError 10061] Kan geen verbinding maken omdat de doelcomputer de verbinding actief heeft geweigerd

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `w`.`calendarDate` cannot be resolved. Did you mean one of the following? [`d`.`calendarDate`, `r`.`endpoint`, `c`.`current`, `r`.`endtime`, `c`.`name`].; line 29 pos 61;
'Project ['r.rideId, 'r.starttime AS ride_starttime#2170, 'r.endtime AS ride_endtime#2171, 'd.dateSK AS date_key#2172, 'c.customerSK AS customer_key#2173, 's1.lockSK AS start_lock_key#2174, 's2.lockSK AS end_lock_key#2175, 'w.weather_category AS weather_key#2176, 'r.vehicleid AS vehicle_id#2177, ('unix_timestamp('r.endtime) - 'unix_timestamp('r.starttime)) AS duration#2178]
+- 'Join Inner, (date_format(starttime#3, yyyy-MM-dd, Some(Europe/Brussels)) = 'w.calendarDate)
   :- Join Inner, (endlockid#8 = cast(lockId#2180 as int))
   :  :- Join Inner, (startlockid#7 = cast(lockId#547 as int))
   :  :  :- Join Inner, ((subscriptionid#6 = userId#517) AND ((starttime#3 >= scd_start#527) AND (starttime#3 <= scd_end#528)))
   :  :  :